In [ ]:
import torch
import torch.autograd as autograd         # computation graph
from torch import Tensor                  # tensor node in the computation graph
import torch.nn as nn                     # neural networks
import torch.optim as optim               # optimizers e.g. gradient descent, ADAM, etc.

import numpy as np
import math
import time
from timeit import default_timer as timer

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as cls
from itertools import product
# Device configuration
device = torch.device('cpu')

In [ ]:
torch.set_default_dtype(torch.float64)

class SNN(nn.Module):

    def __init__(self, layers1,layers2, device):
        super().__init__() #call __init__ from parent class
        self.layers1 = layers1
        self.layers2 = layers2
        self.device = device
        self.activation = nn.Tanh()
        self.loss_function = nn.MSELoss(reduction ='mean')
        self.linears1 = nn.ModuleList([
            nn.Linear(layers1[i], layers1[i+1]).to(dtype=torch.float64) 
            for i in range(len(layers1) - 1)
        ])
        self.linears2 = nn.ModuleList([
            nn.Linear(layers2[i], layers2[i+1]).to(dtype=torch.float64) 
            for i in range(len(layers2) - 1)
        ])
        self.M = layers1[-1]
        self.omega = torch.ones((self.M, 1), dtype=torch.float64).to(device)
        
        self.alpha = 1e-2
        self.miu = torch.tensor(15.0)
        
        self.a = 0
        self.b = 1
        
        self.T = torch.tensor(0.1)
        self.t0 = torch.tensor(0)
        self.nu = torch.tensor(0.01)
        

        'Xavier Normal Initialization'
        for i1 in range(len(layers1)-1):
            nn.init.xavier_normal_(self.linears1[i1].weight.data, gain=1.0)
            nn.init.zeros_(self.linears1[i1].bias.data)
            
        for i2 in range(len(layers2)-1):
            nn.init.xavier_normal_(self.linears2[i2].weight.data, gain=1.0)
            nn.init.zeros_(self.linears2[i2].bias.data)
            
    
        

    'foward pass'
    def forward_y(self,x):

        if torch.is_tensor(x) != True:
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)
        

        l_x = torch.prod((x[:,0][:,None] - self.a) * (self.b - x[:,0][:,None]), 1)[:,None] 
        
        #x = torch.sin( math.pi * x)
        for i in range(len(self.layers1)-2):
            z = self.linears1[i](x)
            x = torch.sin(z)
        
        x = self.linears1[-1](x) #+ x_sum

        output = l_x *  x


        return output
    
    def forward_p(self,x):

        if torch.is_tensor(x) != True:
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)
        

        l_x = torch.prod((x[:,0][:,None] - self.a) * (self.b - x[:,0][:,None]), 1)[:,None] 
        
        #x = torch.sin( math.pi * x)
        for i in range(len(self.layers2)-2):
            z = self.linears2[i](x)
            x = torch.sin(z)
        
        x = self.linears2[-1](x) #+ x_sum

        output = l_x *  x


        return output
                                
    def exact_solution(self, x_t, x_t_0, x_t_T):
        g = x_t
        x, t= x_t[:,0][:,None], x_t[:,-1][:,None]
        x0, xT  = x_t_0[:,0][:,None], x_t_T[:,0][:,None]
        
        y_exact = torch.exp(t) * torch.sin(2 * math.pi * x)
        y_T_exact = torch.exp(self.T) * torch.sin(2 * math.pi * xT)
        y_0_exact = torch.exp(self.t0) * torch.sin(2 * math.pi * x0)
       
        p_exact = - self.alpha * y_exact
        p_0_exact = - self.alpha * y_0_exact
        p_T_exact = - self.alpha * y_T_exact
        
        u_exact = - 1 / self.alpha * p_exact
        u_0_exact = - 1 / self.alpha * p_0_exact
        
        
        y_t_exact = y_exact
        y_x_exact = 2 * math.pi*torch.exp(t) * torch.cos(2 * math.pi * x)
        y_x_x_exact = -4 * math.pi*torch.exp(t)**2 * y_exact
        
        p_t_exact = - self.alpha * y_exact
        p_x_exact = - self.alpha * y_x_exact
        p_x_x_exact = - self.alpha * y_x_x_exact
        
        f =  y_t_exact+ y_exact * y_x_exact - self.nu * y_x_x_exact - u_exact
        
        y_d =   p_t_exact + y_exact * p_x_exact + self.nu * p_x_x_exact + y_exact
        return u_exact, y_exact, p_exact,y_0_exact,p_T_exact, y_d, f
    
    
    def loss(self, x_t, x_0_t, x_t_0, x_t_T):
        g = x_t
        g.requires_grad = True
        

        fi_y = self.forward_y(g)
        fi_p = self.forward_p(g)
        
                
        y = fi_y @ self.omega 
        p = fi_p @ self.omega 
        u = - 1/self.alpha * p
        
        y_0 = self.forward_y(x_t_0)@ self.omega
        p_T = self.forward_p(x_t_T)@ self.omega
        
              
        y_xt = autograd.grad(y, g, torch.ones([g.shape[0], 1]).to(self.device), retain_graph=True, create_graph=True)[0]
        y_x = y_xt[:,[0]]
        y_t = y_xt[:,[-1]] 
        y_x_xt = autograd.grad(y_x, g, torch.ones([g.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        y_x_x = y_x_xt[:,[0]]
        
        
        p_xt = autograd.grad(p, g, torch.ones([g.shape[0], 1]).to(self.device), retain_graph=True, create_graph=True)[0]
        p_x = p_xt[:,[0]]
        p_t = p_xt[:,[-1]] 
        p_x_xt = autograd.grad(p_x, g, torch.ones([g.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        p_x_x = p_x_xt[:,[0]]
        
        u_exact, y_exact, p_exact,y_0_exact,p_T_exact, y_d, f = self.exact_solution(x_t, x_t_0, x_t_T)
        
        
        PDE =  y_t + y * y_x - self.nu * y_x_x - u - f
        PDE_0 = y_0 - y_0_exact
        
        r_p =  p_t + y * p_x  + self.nu * p_x_x   + y - y_d
        r_p_T = p_T - p_T_exact
        
        
        
        

         
        loss_PDE = self.loss_function(PDE, torch.zeros(PDE.shape).to(self.device))
        loss_PDE_0 = self.loss_function(PDE_0, torch.zeros(PDE_0.shape).to(self.device))
        loss_p = self.loss_function(r_p, torch.zeros(r_p.shape).to(self.device))
        loss_p_T = self.loss_function(r_p_T, torch.zeros(r_p_T.shape).to(self.device))
        
    
        return   loss_p, loss_p_T, loss_PDE, loss_PDE_0
       
                                
                                
                                
    
                                
                                
    def train_model_adam(self, optimizer, x_t, x_0_t, x_t_0, x_t_T ,n_epoch):
        if torch.is_tensor(x_t) != True:
            x_t = torch.from_numpy(x_t).to(dtype=torch.float64).to(self.device)

        iter_s = 0
        epsilon = 1e-3   
        gamma = 0.9
        eps = 1e-8
        w_min = 0.3
        w_max = 0.7
        beta = 0.5

        ema_p, ema_PDE = 1.0, 1.0
        initial_loss = None

        while iter_s < n_epoch:
            optimizer.zero_grad()
            loss_p, loss_p_T, loss_PDE, loss_PDE_0 = self.loss(x_t, x_0_t, x_t_0, x_t_T)

           
            val_p = float(loss_p.detach())
            val_PDE = float(loss_PDE.detach())

           
            new_ema_p = gamma * ema_p + (1 - gamma) * val_p
            new_ema_PDE = gamma * ema_PDE + (1 - gamma) * val_PDE

           
            r1 = max((ema_p - new_ema_p) / (ema_p + eps), 0.0)
            r2 = max((ema_PDE - new_ema_PDE) / (ema_PDE + eps), 0.0)

          
            raw_w1 = 1.0 / ((r1 + eps) ** beta)
            raw_w2 = 1.0 / ((r2 + eps) ** beta)

        
            w1 = raw_w1 / (raw_w1 + raw_w2)
            w2 = 1.0 - w1

          
            w1 = w_min + (w_max - w_min) * w1
            w2 = 1.0 - w1

         
            loss = w1*loss_p + w2*loss_PDE
            loss1 = loss_p + loss_p_T + loss_PDE + loss_PDE_0

           
            if iter_s == 0:
                initial_loss = float(loss1.detach())
                if initial_loss < 1e-10:   
                    print("Stop")
                    break

            # 相对下降停止条件
            if float(loss1.detach()) / initial_loss < epsilon:
                print("Stop:", iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()),w1,w2)
                break

           
            loss.backward()
            optimizer.step()

     
            ema_p, ema_PDE = new_ema_p, new_ema_PDE

       
            if iter_s == 0 or iter_s % 100 == 0:
                print(iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()),w1,w2)

            iter_s += 1

# === Sampling ===
def sample_domain(n=32):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 0.1, n)
    xx, yy = np.meshgrid(x, y)
    xy = np.stack([xx.ravel(), yy.ravel()], axis=-1)
    x_train = torch.tensor(xy, dtype=torch.float64)
    return x_train

x = np.linspace(0, 1, 100)
t = np.linspace(0, 0.1, 100)
#print(t)
X, T = np.meshgrid(x , t)

x_t_0 = torch.tensor(list(product(x.tolist(),[0])))
x_t_T = torch.tensor(list(product(x.tolist(),[1])))

x_0_t = torch.tensor(list(product([0,1],t.tolist())))

M = 200
layers1 = np.array([2, 100, 100, 100, 100, M]) 
layers2 = np.array([2, 100, 100, 100, 100, M])

SNN = SNN(layers1, layers2, device)

SNN.to(device)
optimizer = torch.optim.Adam(SNN.parameters(), lr=0.001)
x_t = sample_domain(n=32)
start = timer()
SNN.train_model_adam(optimizer, x_t, x_0_t, x_t_0, x_t_T, 5e3)

In [ ]:
from scipy.optimize import least_squares


phi_y = SNN.forward_y(x_t)
phi_p = SNN.forward_p(x_t)
phi_y_0 = SNN.forward_y(x_t_0).detach().cpu().numpy()
phi_p_T = SNN.forward_p(x_t_T).detach().cpu().numpy()
phi_y_x0 = SNN.forward_y(x_0_t).detach().cpu().numpy()
phi_p_x0 = SNN.forward_p(x_0_t).detach().cpu().numpy()


def compute_derivatives(phi: torch.Tensor, x: torch.Tensor):
    grads = []
    hess_x = []
    for j in range(phi.shape[1]):
        phi_j = phi[:, j:j+1]
        grad = autograd.grad(phi_j, x, torch.ones_like(phi_j), create_graph=True, retain_graph=True)[0]
        dphi_dx, dphi_dt = grad[:, 0:1], grad[:, 1:2]
        d2phi_dx = autograd.grad(dphi_dx, x, torch.ones_like(dphi_dx), create_graph=True, retain_graph=True)[0][:, 0:1]
        grads.append(torch.cat([dphi_dx, dphi_dt], dim=1))
        hess_x.append(d2phi_dx)
    
    grads = torch.stack(grads, dim=2)  # [N, 2, M]
    hess_x = torch.stack(hess_x, dim=2)  # [N, 1, M]
    
    return grads.detach().cpu().numpy(), hess_x.detach().cpu().numpy()


D_y, D_y_dxx = compute_derivatives(phi_y, x_t)
D_p, D_p_dxx = compute_derivatives(phi_p, x_t)

D_y_dx = D_y[:, 0, :]  # shape: [N, M]
D_y_dt = D_y[:, 1, :]
D_y_dxx = D_y_dxx[:, 0, :]

D_p_dx = D_p[:, 0, :]
D_p_dt = D_p[:, 1, :]
D_p_dxx = D_p_dxx[:, 0, :]


_, _, _, y_0_exact, p_T_exact, y_d, f = SNN.exact_solution(x_t,x_t_0,x_t_T)

_, y_bc, p_bc, _ ,_ , _, _ = SNN.exact_solution(x_0_t,x_t_0,x_t_T)


# 转 numpy
Phi_y = phi_y.detach().cpu().numpy()
Phi_p = phi_p.detach().cpu().numpy()
y_bc_np = y_bc.detach().cpu().numpy()
p_bc_np = p_bc.detach().cpu().numpy()
f_np = f.detach().cpu().numpy()
y_d_np = y_d.detach().cpu().numpy()
y_0_np = y_0_exact.detach().cpu().numpy()
p_T_np = p_T_exact.detach().cpu().numpy()


omega_y = np.ones((M, 1))
omega_p = np.ones((M, 1))
alpha = SNN.alpha
nu = SNN.nu.item()

def residual_y(omega_flat):
    omega = omega_flat[:, None]
    y = Phi_y @ omega
    p = Phi_p @ omega_p
    dy_dx = D_y_dx @ omega
    dy_dt = D_y_dt @ omega
    dy_dxx = D_y_dxx @ omega
    u = - 1 / alpha * p
    r_pde = dy_dt + y * dy_dx - nu * dy_dxx - u - f_np
    r_pde2 = phi_y_0 @ omega - y_0_np
    r_bc =  phi_y_x0 @ omega - y_bc_np
    return np.vstack([r_pde, r_pde2 , r_bc]).ravel()

def residual_p(omega_flat):
    omega = omega_flat[:, None]
    y = Phi_y @ omega_y
    p = Phi_p @ omega
    dp_dx = D_p_dx @ omega
    dp_dt = D_p_dt @ omega
    dp_dxx = D_p_dxx @ omega
    r_pde = dp_dt + y * dp_dx + nu * dp_dxx + y - y_d_np
    r_pde2 = phi_p_T @ omega - p_T_np
    r_bc =  phi_p_x0 @ omega - p_bc_np
    return np.vstack([r_pde, r_pde2, r_bc]).ravel()




In [ ]:

epoch = 20
for i in range(epoch):
    res_y = least_squares(residual_y, omega_y.ravel(), method='trf')
    omega_y = res_y.x[:, None]
    res_p = least_squares(residual_p, omega_p.ravel(), method='trf')
    omega_p = res_p.x[:, None]
    print(f"🔁 Iter {i+1}/{epoch}, state: {res_y.cost:.2e}, adjoint: {res_p.cost:.2e}")

end = timer()
print("consumed time: " + str(end - start) + "s")

In [ ]:
omega_p = torch.tensor(omega_p, dtype=torch.float64).to(device)
omega_y = torch.tensor(omega_y, dtype=torch.float64).to(device)
def test(x_test, x_0_t, x_t_0, x_t_T):

        if torch.is_tensor(x_test) != True:
            x_test = torch.from_numpy(x_test).to(dtype=torch.float64).to(self.device)
            
        u, y, p, _,_,_,_ = SNN.exact_solution(x_test, x_t_0,x_t_T)

        phi_pred_y = SNN.forward_y(x_test)
        phi_pred_p = SNN.forward_p(x_test)
        y_pred = phi_pred_y @ omega_y
        p_pred = phi_pred_p @ omega_p
        u_pred = -1/SNN.alpha * p_pred
        #print(SNN.alpha)
    
        error_y = torch.linalg.norm((y - y_pred),2)/torch.linalg.norm(y,2)
        y_pred = y_pred.cpu().detach().numpy()
        
        error_p = torch.linalg.norm((p - p_pred),2)/torch.linalg.norm(p,2)
        p_pred = p_pred.cpu().detach().numpy()

        error_u = torch.linalg.norm((u - u_pred),2)/torch.linalg.norm(u,2)
        u_pred = u_pred.cpu().detach().numpy()
        
    
        return error_y, y_pred, y, error_p, p_pred, p,error_u, u_pred, u

In [ ]:
x_test = sample_domain(n=100).to(device)
error_y, y_pred, y ,error_p, p_pred, p,error_u, u_pred, u= test(x_test, x_0_t, x_t_0, x_t_T)
print('Test Error y: {:.2e}'.format(error_y))
#print('Test Error p: {:.2e}'.format(error_p))
print('Test Error u: {:.2e}'.format(error_u))